## Preparation

In [1]:
# import torch
# from transformers.activations import ACT2FN
from transformer_lens import HookedTransformer
import examples
from neuron_analysis import neuron_analysis

In [2]:
import importlib

In [3]:
importlib.reload(examples)

<module 'examples' from '/mounts/work/sgerstner/RW_functionalities/examples.py'>

In [2]:
DEVICE='cuda:0'

In [3]:
args = examples.create_args(
    neuron_subset_name="weakening",
    intervention_type="zero_ablation",
    metric="entropy",
    topk=1,
    device=DEVICE,
    gate='-',
    post='+',
    #data_dir="../RW_functionalities_results",
)

In [4]:
model = HookedTransformer.from_pretrained(
    'allenai/OLMo-7B-0424-hf',
    #refactor_glu=True,
    device=DEVICE
)

`torch_dtype` is deprecated! Use `dtype` instead!


Loading weights:   0%|          | 0/226 [00:00<?, ?it/s]

Loaded pretrained model allenai/OLMo-7B-0424-hf into HookedTransformer


In [5]:
input_ids = model.to_tokens(
    "Yesterday (21 December) the Government announced a package of support for hospitality and leisure businesses that are losing trade because of the Omic"
)#.flatten()

In [6]:
neuron_list = examples.find_neurons(args)

In [7]:
result_str, cache_clean, cache_ablated = examples.show_single_text(
    args=args,
    model=model,
    input_ids=input_ids,
    pos=input_ids.shape[1]-2,#ignore the last token, which is the ground-truth output
    neuron_list=neuron_list,
    return_cache=True,
)

running clean model...


dict_keys(['hook_embed', 'blocks.0.hook_resid_pre', 'blocks.0.ln1.hook_scale', 'blocks.0.ln1.hook_normalized', 'blocks.0.attn.hook_q', 'blocks.0.attn.hook_k', 'blocks.0.attn.hook_v', 'blocks.0.attn.hook_rot_q', 'blocks.0.attn.hook_rot_k', 'blocks.0.attn.hook_attn_scores', 'blocks.0.attn.hook_pattern', 'blocks.0.attn.hook_z', 'blocks.0.hook_attn_out', 'blocks.0.hook_resid_mid', 'blocks.0.ln2.hook_scale', 'blocks.0.ln2.hook_normalized', 'blocks.0.mlp.hook_pre', 'blocks.0.mlp.hook_pre_linear', 'blocks.0.mlp.hook_post', 'blocks.0.hook_mlp_out', 'blocks.0.hook_resid_post', 'blocks.1.hook_resid_pre', 'blocks.1.ln1.hook_scale', 'blocks.1.ln1.hook_normalized', 'blocks.1.attn.hook_q', 'blocks.1.attn.hook_k', 'blocks.1.attn.hook_v', 'blocks.1.attn.hook_rot_q', 'blocks.1.attn.hook_rot_k', 'blocks.1.attn.hook_attn_scores', 'blocks.1.attn.hook_pattern', 'blocks.1.attn.hook_z', 'blocks.1.hook_attn_out', 'blocks.1.hook_resid_mid', 'blocks.1.ln2.hook_scale', 'blocks.1.ln2.hook_normalized', 'blocks.1.m

In [8]:
print(result_str)

Input tokens:
["<|endoftext|>", "Yesterday", " (", "21", " December", ")", " the", " Government", " announced", " a", " package", " of", " support", " for", " hospitality", " and", " leisure", " businesses", " that", " are", " losing", " trade", " because", " of", " the", " O", "mic"]
Ground-truth output token:
mic

The clean model outputs:
mic
with score 21.1351261138916

The ablated model outputs:


with score 9.471395492553711
top tokens promoted by the neurons (overall effect):
["mic", "MIC", "Mic", "MI", "micro", "Micro", " Mic", "PEC", "NS", "mi", " mic", "yster", "ECD", "mn", "VO", "microm"]
top tokens suppressed by the neurons (overall effect):
[" the", " one", " old", " a", " older", " many", " individual", " make", " it", " them", " our", " their", " big", " others", " other", " early"]


In [9]:
(model.ln_final(cache_clean['blocks.31.hook_resid_post'][0,-1])@model.W_U.detach()[:,model.to_single_token('mic')]).item()

21.135128021240234

In [10]:
(model.ln_final(cache_ablated['blocks.31.hook_resid_post'][0,-1])@model.W_U.detach()[:,model.to_single_token('mic')]).item()

1.3977943658828735

score difference to explain, *without* layer norm:

In [11]:
((cache_clean['blocks.31.hook_resid_post'][0,-1]-cache_ablated['blocks.31.hook_resid_post'][0,-1])@model.W_U.detach()[:,model.to_single_token('mic')]).item()

3.579124927520752

In [12]:
print("Re-running clean model...")
logits_clean_new, cache_clean_new = model.run_with_cache(input_ids[:,:input_ids.shape[1]-1])
print(cache_clean_new[f'blocks.31.hook_mlp_out'][0:1,-1:])

Re-running clean model...
tensor([[[ 0.0374, -0.0318, -0.2299,  ..., -0.1227,  0.1623, -0.1698]]],
       device='cuda:0')


Note: I suspect llmtt computes the scores *with* layer norm, hence a mismatch.

## Neuron 31.2578
we already know from llmtt this neuron is relevant.

In [13]:
neuron_analysis(model, 31, 2578)

tensor([[[ 0.0374, -0.0318, -0.2299,  ..., -0.1227,  0.1623, -0.1698]]],
       device='cuda:0')
tensor([[[ 0.0374, -0.0318, -0.2299,  ..., -0.1227,  0.1623, -0.1698]]],
       device='cuda:0')
tensor([[[ 0.0374, -0.0318, -0.2299,  ..., -0.1227,  0.1623, -0.1698]]],
       device='cuda:0')
tensor([[[ 0.0486, -0.0378, -0.8705,  ..., -0.2270, -0.0013, -1.3032]]],
       device='cuda:0')


/mounts/work/sgerstner/RW_functionalities/src/weight_analysis_utils/utils.py:310: SyntaxWarning: invalid escape sequence '\l'
  new_keys[i] : f"{string_keys[i]} $\leq$ cos < {1 if i+1==len(keys) else string_keys[i+1]}"
/mounts/work/sgerstner/RW_functionalities/src/weight_analysis_utils/utils.py:310: SyntaxWarning: invalid escape sequence '\l'
  new_keys[i] : f"{string_keys[i]} $\leq$ cos < {1 if i+1==len(keys) else string_keys[i+1]}"
/mounts/work/sgerstner/RW_functionalities/src/weight_analysis_utils/utils.py:310: SyntaxWarning: invalid escape sequence '\l'
  new_keys[i] : f"{string_keys[i]} $\leq$ cos < {1 if i+1==len(keys) else string_keys[i+1]}"
/mounts/work/sgerstner/RW_functionalities/src/weight_analysis_utils/utils.py:310: SyntaxWarning: invalid escape sequence '\l'
  new_keys[i] : f"{string_keys[i]} $\leq$ cos < {1 if i+1==len(keys) else string_keys[i+1]}"


gate vs. linear similarity: 0.19408632814884186
gate vs. out similarity: -0.45624828338623047
lin vs. out similarity: -0.41817840933799744
most similar tokens for w_out:
        dot product
Kem        0.070418
Newman     0.055059
Dodge      0.053756
Salem      0.049722
Fors       0.049218
Pho        0.048548
Mang       0.046293
Ri         0.046200
Karn       0.045719
Prest      0.045437
most similar tokens for -w_out:
              dot product
 micro           0.624942
 Micro           0.579665
micro            0.534182
 mic             0.498626
 Mic             0.494360
Micro            0.489219
mic              0.443438
Mic              0.409216
 microm          0.390585
 microscopic     0.350804
most similar tokens for w_in:
             dot product
 micro          0.427889
 Micro          0.375135
micro           0.290706
Micro           0.288925
 mic            0.236249
 Mic            0.218860
 microp         0.194707
 microm         0.185191
Mic             0.175375
 microphone 

In [21]:
print("Re-running clean model...")
logits_clean_new, cache_clean_new = model.run_with_cache(input_ids[:,:input_ids.shape[1]-1])
print(cache_clean_new[f'blocks.31.hook_mlp_out'][0:1,-1:])

Re-running clean model...
tensor([[[ 0.0530, -0.0402, -1.1237,  ..., -0.2683, -0.0660, -1.7512]]],
       device='cuda:0')


How is w_in related to 'mic'?

In [22]:
model.blocks[31].mlp.W_in.detach()[:,2578]@model.W_U.detach()[:,model.to_single_token('mic')]

tensor(0.1380, device='cuda:0')

when activated positively, gate and in both detect 'mic' and write minus 'mic' (next to other similar things)

In [23]:
print("Re-running clean model...")
logits_clean_new, cache_clean_new = model.run_with_cache(input_ids[:,:input_ids.shape[1]-1])
print(cache_clean_new[f'blocks.31.hook_mlp_out'][0:1,-1:])

Re-running clean model...
tensor([[[ 0.0530, -0.0402, -1.1237,  ..., -0.2683, -0.0660, -1.7512]]],
       device='cuda:0')


In [24]:
for subkey in ['pre', 'pre_linear', 'post']:
    print(subkey)
    print('clean')
    print(cache_clean[f'blocks.31.mlp.hook_{subkey}'][0,-1,2578].item())#batch pos neuron
    print('ablated')
    print(cache_ablated[f'blocks.31.mlp.hook_{subkey}'][0,-1,2578].item())
    print('\n')

pre
clean
5.226315498352051
ablated
0.5835620760917664


pre_linear
clean
-1.5932304859161377
ablated
-0.03865267336368561


post
clean
-8.282222747802734
ablated
-0.01447854470461607




In the clean run, the gate detects 'mic' but 'in' does not, leading to a strengthening of 'mic'. In the ablated run, activations are in the same quadrant but much smaller.

In [25]:
print("Re-running clean model...")
logits_clean_new, cache_clean_new = model.run_with_cache(input_ids[:,:input_ids.shape[1]-1])
print(cache_clean_new[f'blocks.31.hook_mlp_out'][0:1,-1:])

Re-running clean model...
tensor([[[ 0.0530, -0.0402, -1.1237,  ..., -0.2683, -0.0660, -1.7512]]],
       device='cuda:0')


In the clean run compared to the ablated run, the neuron increases the mic logit by approx. 8.28*0.443 = ...

In [26]:
(
    cache_clean['blocks.31.mlp.hook_post'][0,-1,2578] - cache_ablated['blocks.31.mlp.hook_post'][0,-1,2578]
).item() * (
    model.blocks[31].mlp.W_out[2578,:].detach()@(model.W_U.detach()[:,model.to_single_token('mic')])
).item()

3.666234302856992

This completely explains the score difference!

In [27]:
print("Re-running clean model...")
logits_clean_new, cache_clean_new = model.run_with_cache(input_ids[:,:input_ids.shape[1]-1])
print(cache_clean_new[f'blocks.31.hook_mlp_out'][0:1,-1:])

Re-running clean model...
tensor([[[ 0.0530, -0.0402, -1.1237,  ..., -0.2683, -0.0660, -1.7512]]],
       device='cuda:0')


What about the residual stream before the last MLP?

In [28]:
(
    (
    cache_clean['blocks.31.hook_resid_mid'][0,-1] - cache_ablated['blocks.31.hook_resid_mid'][0,-1]
    ) @ (
        model.W_U.detach()[:,model.to_single_token('mic')]
    )
).item()

0.7509428262710571

So it's really just the neuron.

In [29]:
print("Re-running clean model...")
logits_clean_new, cache_clean_new = model.run_with_cache(input_ids[:,:input_ids.shape[1]-1])
print(cache_clean_new[f'blocks.31.hook_mlp_out'][0:1,-1:])

Re-running clean model...
tensor([[[ 0.0530, -0.0402, -1.1237,  ..., -0.2683, -0.0660, -1.7512]]],
       device='cuda:0')


Next big question: What causes this neuron to activate in one run and not the other?

## Going further back

### Debugging

In [30]:
model.reset_hooks(including_permanent=True)

In [31]:
print("Re-running clean model...")
logits_clean_new, cache_clean_new = model.run_with_cache(input_ids[:,:input_ids.shape[1]-1])
print(cache_clean_new[f'blocks.31.hook_mlp_out'][0:1,-1:])

Re-running clean model...
tensor([[[ 0.0530, -0.0402, -1.1237,  ..., -0.2683, -0.0660, -1.7512]]],
       device='cuda:0')


In [32]:
for layer in range(32):
    for hook in ["hook_resid_pre", "hook_resid_mid", "ln2.hook_normalized", "mlp.hook_pre", "mlp.hook_pre_linear", "mlp.hook_post", "hook_mlp_out", "hook_resid_post"]:
        diff = (
            cache_clean_new[f"blocks.{layer}.{hook}"][0:1,-1:] - cache_clean[f"blocks.{layer}.{hook}"][0:1,-1:]
        ).abs().max().item()
        if diff!=0:
            print(layer, hook)
            print(diff)

31 mlp.hook_pre
1.837803840637207
31 mlp.hook_pre_linear
3.3975579738616943
31 mlp.hook_post
26.94317626953125
31 hook_mlp_out
2.1006782054901123
31 hook_resid_post
2.100677967071533


In [18]:
print(model.blocks[31].apply_mlp(cache_clean[f'blocks.31.ln2.hook_normalized'][0:1,-1:]))
print(cache_clean[f'blocks.31.hook_mlp_out'][0:1,-1:])
print(cache_clean_new[f'blocks.31.hook_mlp_out'][0:1,-1:])

tensor([[[ 0.0374, -0.0318, -0.2299,  ..., -0.1227,  0.1623, -0.1698]]],
       device='cuda:0')
tensor([[[ 0.0374, -0.0318, -0.2299,  ..., -0.1227,  0.1623, -0.1698]]],
       device='cuda:0')
tensor([[[ 0.0374, -0.0318, -0.2299,  ..., -0.1227,  0.1623, -0.1698]]],
       device='cuda:0')


In [20]:
x = cache_clean_new["blocks.31.ln2.hook_normalized"][0:1, -1:]
y = model.blocks[31].apply_mlp(x)
# print("x shape", x.shape)
# print("y shape", y.shape)
# print("cached shape", cache_clean_new["blocks.31.hook_mlp_out"][0:1,-1:].shape)
print("allclose full slice", torch.allclose(y, cache_clean_new["blocks.31.hook_mlp_out"][0:1,-1:], atol=1e-6, rtol=1e-4))
print("max abs diff full slice", (y - cache_clean_new["blocks.31.hook_mlp_out"][0:1,-1:]).abs().max().item())
y1 = model.blocks[31].apply_mlp(cache_clean_new["blocks.31.ln2.hook_normalized"][0,-1])
# print("y1 shape", y1.shape)
print("allclose 1d vs cached 1d", torch.allclose(y1, cache_clean_new["blocks.31.hook_mlp_out"][0,-1], atol=1e-6, rtol=1e-4))
print("max abs diff 1d", (y1 - cache_clean_new["blocks.31.hook_mlp_out"][0,-1]).abs().max().item())


allclose full slice True
max abs diff full slice 6.407499313354492e-07
allclose 1d vs cached 1d True
max abs diff 1d 6.407499313354492e-07


In [21]:
print(y)
print(y1)
print(cache_clean_new["blocks.31.hook_mlp_out"][0:1,-1:])

tensor([[[ 0.0374, -0.0318, -0.2299,  ..., -0.1227,  0.1623, -0.1698]]],
       device='cuda:0')
tensor([ 0.0374, -0.0318, -0.2299,  ..., -0.1227,  0.1623, -0.1698],
       device='cuda:0')
tensor([[[ 0.0374, -0.0318, -0.2299,  ..., -0.1227,  0.1623, -0.1698]]],
       device='cuda:0')


In [22]:
model.hook_dict

{'hook_embed': HookPoint(),
 'blocks.0.ln1.hook_scale': HookPoint(),
 'blocks.0.ln1.hook_normalized': HookPoint(),
 'blocks.0.ln2.hook_scale': HookPoint(),
 'blocks.0.ln2.hook_normalized': HookPoint(),
 'blocks.0.attn.hook_k': HookPoint(),
 'blocks.0.attn.hook_q': HookPoint(),
 'blocks.0.attn.hook_v': HookPoint(),
 'blocks.0.attn.hook_z': HookPoint(),
 'blocks.0.attn.hook_attn_scores': HookPoint(),
 'blocks.0.attn.hook_pattern': HookPoint(),
 'blocks.0.attn.hook_result': HookPoint(),
 'blocks.0.attn.hook_rot_k': HookPoint(),
 'blocks.0.attn.hook_rot_q': HookPoint(),
 'blocks.0.mlp.hook_pre': HookPoint(),
 'blocks.0.mlp.hook_pre_linear': HookPoint(),
 'blocks.0.mlp.hook_post': HookPoint(),
 'blocks.0.hook_attn_in': HookPoint(),
 'blocks.0.hook_q_input': HookPoint(),
 'blocks.0.hook_k_input': HookPoint(),
 'blocks.0.hook_v_input': HookPoint(),
 'blocks.0.hook_mlp_in': HookPoint(),
 'blocks.0.hook_attn_out': HookPoint(),
 'blocks.0.hook_mlp_out': HookPoint(),
 'blocks.0.hook_resid_pre': H

In [36]:
for layer in range(32):
    print(layer)
    for hook in ["hook_resid_pre", "hook_resid_mid", "ln2.hook_normalized", "mlp.hook_pre", "mlp.hook_pre_linear", "mlp.hook_post", "hook_mlp_out", "hook_resid_post"]:
        diff = (
            cache_clean_new[f"blocks.{layer}.{hook}"][0:1,-1:] - cache_ablated[f"blocks.{layer}.{hook}"][0:1,-1:]
        ).abs().max().item()
        if diff!=0:
            print(hook)
            print(diff)

0
mlp.hook_post
0.05847415328025818
hook_mlp_out
0.002836465835571289
hook_resid_post
0.0028367042541503906
1
hook_resid_pre
0.0028367042541503906
hook_resid_mid
0.00655364990234375
ln2.hook_normalized
0.037700653076171875
mlp.hook_pre
0.10508739948272705
mlp.hook_pre_linear
0.07043245434761047
mlp.hook_post
0.30814191699028015
hook_mlp_out
0.021032094955444336
hook_resid_post
0.027585744857788086
2
hook_resid_pre
0.027585744857788086
hook_resid_mid
0.020996034145355225
ln2.hook_normalized
0.29217836260795593
mlp.hook_pre
0.21028509736061096
mlp.hook_pre_linear
0.19647330045700073
mlp.hook_post
0.09204387664794922
hook_mlp_out
0.016140341758728027
hook_resid_post
0.008361462503671646
3
hook_resid_pre
0.008361462503671646
hook_resid_mid
0.008488625288009644
ln2.hook_normalized
0.11279749870300293
mlp.hook_pre
0.2312116175889969
mlp.hook_pre_linear
0.19661694765090942
mlp.hook_post
0.06320451945066452
hook_mlp_out
0.004917312413454056
hook_resid_post
0.009377241134643555
4
hook_resid_pre

In [25]:
neuron_list

[(tensor(0, device='cuda:7'), tensor(365, device='cuda:7')),
 (tensor(0, device='cuda:7'), tensor(3234, device='cuda:7')),
 (tensor(0, device='cuda:7'), tensor(3467, device='cuda:7')),
 (tensor(0, device='cuda:7'), tensor(5769, device='cuda:7')),
 (tensor(0, device='cuda:7'), tensor(10043, device='cuda:7')),
 (tensor(1, device='cuda:7'), tensor(130, device='cuda:7')),
 (tensor(1, device='cuda:7'), tensor(227, device='cuda:7')),
 (tensor(1, device='cuda:7'), tensor(1539, device='cuda:7')),
 (tensor(1, device='cuda:7'), tensor(1720, device='cuda:7')),
 (tensor(1, device='cuda:7'), tensor(2192, device='cuda:7')),
 (tensor(1, device='cuda:7'), tensor(2283, device='cuda:7')),
 (tensor(1, device='cuda:7'), tensor(2328, device='cuda:7')),
 (tensor(1, device='cuda:7'), tensor(2638, device='cuda:7')),
 (tensor(1, device='cuda:7'), tensor(3983, device='cuda:7')),
 (tensor(1, device='cuda:7'), tensor(3985, device='cuda:7')),
 (tensor(1, device='cuda:7'), tensor(4101, device='cuda:7')),
 (tensor(1

In [20]:
model.blocks[31].mlp.b_in[2578]

tensor(0., device='cuda:7', requires_grad=True)

In [26]:
model.blocks[31].ln2

LayerNormPre(
  (hook_scale): HookPoint()
  (hook_normalized): HookPoint()
)

In [29]:
# def hypothetical_neuron_activation(residual):
#         #normalised_residual = residual
#         normalised_residual = model.blocks[31].ln2(residual)
#         return (ACT2FN[model.cfg.act_fn](normalised_residual @ model.blocks[31].mlp.W_gate[:,2578]) * (normalised_residual @ model.blocks[31].mlp.W_in[:,2578])).item()

def hypothetical_ln(residual):
    return model.blocks[31].ln2(residual)

def hypothetical_hook_pre(x):
    return x @ model.blocks[31].mlp.W_gate[:,2578]

def hypothetical_hook_pre_linear(x):
    return x @ model.blocks[31].mlp.W_in[:,2578]

def act_from_branches(pre_act, pre_linear):
    ans = model.blocks[31].mlp.act_fn(pre_act) * pre_linear + model.blocks[31].mlp.b_in[2578]
    if isinstance(ans, torch.Tensor):
        return ans.item()
    else:
        return ans

def hypothetical_neuron_activation(x, verbose=True, with_ln=True):
    if with_ln:
        x = hypothetical_ln(x)
    pre_act = hypothetical_hook_pre(x)
    pre_linear = hypothetical_hook_pre_linear(x)
    post = act_from_branches(pre_act=pre_act, pre_linear=pre_linear)
    if verbose:
        print(f'>> pre: {pre_act}')
        print(f'>> pre_linear: {pre_linear}')
        print(f'>> post: {post}')
    return post


In [27]:
def show_hypothetical_activations(layer, loc, **kwargs):
    print('> clean:')
    hypothetical_clean_activation = hypothetical_neuron_activation(cache_clean[f'blocks.{layer}.hook_resid_{loc}'][0,-1], **kwargs)
    print('> ablated:')
    hypothetical_ablated_activation = hypothetical_neuron_activation(cache_ablated[f'blocks.{layer}.hook_resid_{loc}'][0,-1], **kwargs)
    print("> diff:", hypothetical_clean_activation - hypothetical_ablated_activation)

In [29]:
print(
    "originally computed normalised residual:", cache_clean[f'blocks.31.ln2.hook_normalized'][0,-1]
)
print(
    "recomputed:", hypothetical_ln(cache_clean['blocks.31.hook_resid_mid'][0,-1])
)

originally computed normalised residual: tensor([-1.0645,  1.1939, -0.0757,  ...,  0.4562, -1.2905,  0.7178],
       device='cuda:7')
recomputed: tensor([-1.0645,  1.1939, -0.0757,  ...,  0.4562, -1.2905,  0.7178],
       device='cuda:7')


In [30]:
torch.max(torch.abs(cache_clean[f'blocks.31.ln2.hook_normalized'][0,-1]-hypothetical_ln(cache_clean['blocks.31.hook_resid_mid'][0,-1])))

tensor(7.4506e-09, device='cuda:7')

In [38]:
act_from_branches(
    pre_act=cache_clean[f'blocks.31.mlp.hook_pre'][0,-1,2578],
    pre_linear=cache_clean[f'blocks.31.mlp.hook_pre_linear'][0,-1,2578],
)

-8.282222747802734

In [36]:
hypothetical_hook_pre(cache_clean[f'blocks.31.ln2.hook_normalized'][0,-1])

tensor(7.0641, device='cuda:7')

In [42]:
model.cfg

HookedTransformerConfig:
{'NTK_by_parts_factor': 8.0,
 'NTK_by_parts_high_freq_factor': 4.0,
 'NTK_by_parts_low_freq_factor': 1.0,
 'NTK_original_ctx_len': 8192,
 'act_fn': 'silu',
 'attention_dir': 'causal',
 'attn_only': False,
 'attn_scale': np.float64(11.313708498984761),
 'attn_scores_soft_cap': -1.0,
 'attn_types': None,
 'checkpoint_index': None,
 'checkpoint_label_type': None,
 'checkpoint_value': None,
 'd_head': 128,
 'd_mlp': 11008,
 'd_model': 4096,
 'd_vocab': 50304,
 'd_vocab_out': 50304,
 'decoder_start_token_id': None,
 'default_prepend_bos': True,
 'device': 'cuda:7',
 'dtype': torch.float32,
 'eps': 1e-05,
 'experts_per_token': None,
 'final_rms': False,
 'from_checkpoint': False,
 'gated_mlp': True,
 'init_mode': 'gpt2',
 'init_weights': False,
 'initializer_range': np.float64(0.0125),
 'load_in_4bit': False,
 'model_name': 'OLMo-7B-0424-hf',
 'n_ctx': 4096,
 'n_devices': 1,
 'n_heads': 32,
 'n_key_value_heads': None,
 'n_layers': 32,
 'n_params': 6476005376,
 'norma

In [43]:
model.cfg.is_layer_norm_activation()

False

In [37]:
model.blocks

ModuleList(
  (0-31): 32 x TransformerBlock(
    (ln1): LayerNormPre(
      (hook_scale): HookPoint()
      (hook_normalized): HookPoint()
    )
    (ln2): LayerNormPre(
      (hook_scale): HookPoint()
      (hook_normalized): HookPoint()
    )
    (attn): Attention(
      (hook_k): HookPoint()
      (hook_q): HookPoint()
      (hook_v): HookPoint()
      (hook_z): HookPoint()
      (hook_attn_scores): HookPoint()
      (hook_pattern): HookPoint()
      (hook_result): HookPoint()
      (hook_rot_k): HookPoint()
      (hook_rot_q): HookPoint()
    )
    (mlp): GatedMLP(
      (hook_pre): HookPoint()
      (hook_pre_linear): HookPoint()
      (hook_post): HookPoint()
    )
    (hook_attn_in): HookPoint()
    (hook_q_input): HookPoint()
    (hook_k_input): HookPoint()
    (hook_v_input): HookPoint()
    (hook_mlp_in): HookPoint()
    (hook_attn_out): HookPoint()
    (hook_mlp_out): HookPoint()
    (hook_resid_pre): HookPoint()
    (hook_resid_mid): HookPoint()
    (hook_resid_post): HookP

In [ ]:
show_hypothetical_activations(31, "mid", with_ln=True)

In [25]:
print("with ln:")
show_hypothetical_activations(31, "mid", with_ln=True)
print("without ln:")
show_hypothetical_activations(31, "mid", with_ln=False)

with ln:
> clean:
>> pre: 7.064119338989258
>> pre_linear: -4.990790367126465
>> post: -35.22541427612305
> ablated:
>> pre: 0.7887684106826782
>> pre_linear: -0.12108016014099121
>> post: -0.06566552072763443
> diff: -35.15974875539541
without ln:
> clean:
>> pre: 0.953421950340271
>> pre_linear: -0.6735912561416626
>> post: -0.46355384588241577
> ablated:
>> pre: 0.852280855178833
>> pre_linear: -0.13082972168922424
>> post: -0.07816912978887558
> diff: -0.3853847160935402


### Actual continuation

In [30]:
for layer in reversed(range(32)):
    for loc in ["mid", "pre"]:
        print(f"layer {layer}, {loc}:")
        show_hypothetical_activations(layer, loc)

layer 31, mid:
> clean:
>> pre: 7.064119338989258
>> pre_linear: -4.990790367126465
>> post: -35.22541427612305
> ablated:
>> pre: 0.7887684106826782
>> pre_linear: -0.12108016014099121
>> post: -0.06566552072763443
> diff: -35.15974875539541
layer 31, pre:
> clean:
>> pre: 6.672290802001953
>> pre_linear: -4.69473934173584
>> post: -31.28507423400879
> ablated:
>> pre: 6.036002159118652
>> pre_linear: -1.3221402168273926
>> post: -7.961404800415039
> diff: -23.32366943359375
layer 30, mid:
> clean:
>> pre: 4.686041831970215
>> pre_linear: -3.5700809955596924
>> post: -16.57666015625
> ablated:
>> pre: 4.7347092628479
>> pre_linear: -3.437246561050415
>> post: -16.132638931274414
> diff: -0.44402122497558594
layer 30, pre:
> clean:
>> pre: 4.739480018615723
>> pre_linear: -4.075556755065918
>> post: -19.14859962463379
> ablated:
>> pre: 4.936550617218018
>> pre_linear: -1.5847150087356567
>> post: -7.7672624588012695
> diff: -11.38133716583252
layer 29, mid:
> clean:
>> pre: 3.17662715